In [1]:
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, random_split

from dataset import SingleNTrajectoryDataset

In [2]:
# This works when the notebook's working directory is att_N8/
data_dir = Path("../GenerateTraj/data/sz_N8").resolve()

files = sorted(data_dir.glob("trajectory_*.npy"))

if not files:
    raise FileNotFoundError(f"No trajectories found in {data_dir}")

sz = np.stack([np.load(file) for file in files]).astype(np.float32)

print("Number of files:", len(files))
print("Data shape:", sz.shape)
print("Data type:", sz.dtype)
print("First values:", sz[0, :5])

Number of files: 1000
Data shape: (1000, 210)
Data type: float32
First values: [-1.         -0.742613   -0.7532236  -0.7589154  -0.75182694]


In [3]:
dataset = SingleNTrajectoryDataset(sz)

features, delta_sz = dataset[0]

print(features.shape)  # torch.Size([209, 2])
print(delta_sz.shape)  # torch.Size([209])
print(features[:3])

torch.Size([209, 2])
torch.Size([209])
tensor([[-1.0000,  0.0000],
        [-0.7426,  0.0048],
        [-0.7532,  0.0096]])


In [4]:
train_size = 800
val_size = 100
test_size = len(dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_set, val_set, test_set = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator,
)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [5]:
x_batch, delta_batch = next(iter(train_loader))

print(x_batch.shape)      # torch.Size([64, 209, 2])
print(delta_batch.shape)  # torch.Size([64, 209])

torch.Size([64, 209, 2])
torch.Size([64, 209])
